In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

In [3]:
df.shape

(2000, 8)

In [4]:
df.head(3)

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [5]:
df.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [6]:
df=df.drop('id',axis=1)

In [7]:
answer=df['answer']
df=df.drop('answer',axis=1)

In [8]:
df.shape

(2000, 6)

In [9]:
import re

def clean_text(text):
    text=str(text).lower()
    text=re.sub(r'[^a-zA-Z0-9\s]',' ',text)
    text=re.sub(r'\s+',' ',text).strip()
    return text

df['prompt']=df['prompt'].apply(clean_text)
df['A']=df['A'].apply(clean_text)
df['B']=df['B'].apply(clean_text)
df['C']=df['C'].apply(clean_text)
df['D']=df['D'].apply(clean_text)
df['E']=df['E'].apply(clean_text)

In [10]:
from sklearn.model_selection import train_test_split

x_train,x_val,y_train,y_val=train_test_split(df,answer,train_size=0.8,random_state=1)

In [11]:
def score_to_labels(scores,top1=False):
    labels=[]
    top1_labels=[]
    for score in scores:
        scored=np.argsort(score)[::-1][:3]
        top_3=[chr(65+i) for i in scored]
        labels.append(" ".join(top_3))
        top1_labels.append(chr(65+np.argsort(score)[::-1][0]))
    if not top1:
        return labels
    else:
        return top1_labels

In [12]:
def map_at_3(predictions,ground_truth):
    ap_scores=[]
    for pred, truth in zip(predictions, ground_truth):
        pred_list = pred.split()
        if truth in pred_list:
            ap_scores.append(1.0 / (pred_list.index(truth) + 1))
        else:
            ap_scores.append(0.0)
    return sum(ap_scores) / len(ap_scores)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf=TfidfVectorizer(sublinear_tf=True)
all_text=df['prompt'].tolist()+df['A'].tolist()+df['B'].tolist()+df['C'].tolist()+df['D'].tolist()+df['E'].tolist()
tfidf.fit(all_text)

TfidfVectorizer(sublinear_tf=True)

In [14]:
tfidf_train_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(x_train['prompt']+" "+x_train[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_train_scores.append(np.asarray(scores).flatten())
tfidf_train_scores=np.array(tfidf_train_scores).T

In [15]:
tfidf_val_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(x_val['prompt']+" "+x_val[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_val_scores.append(np.asarray(scores).flatten())
tfidf_val_scores=np.array(tfidf_val_scores).T

In [16]:
tfidf_train_labels=score_to_labels(tfidf_train_scores,top1=False)
tfidf_train_map3=map_at_3(tfidf_train_labels,y_train)
print(tfidf_train_map3)

tfidf_val_labels=score_to_labels(tfidf_val_scores,top1=False)
tfidf_val_map3=map_at_3(tfidf_val_labels,y_val)
print(tfidf_val_map3)

0.524375
0.5133333333333333


In [17]:
from sklearn.metrics import accuracy_score,f1_score

def support_metrics(train_scores,val_scores):
    train_acc=accuracy_score(score_to_labels(train_scores,top1=True),y_train)
    val_acc=accuracy_score(score_to_labels(val_scores,top1=True),y_val)
    train_f1=f1_score(score_to_labels(train_scores,top1=True),y_train,average='macro')
    val_f1=f1_score(score_to_labels(val_scores,top1=True),y_val,average='macro')
    return train_acc,val_acc,train_f1,val_f1
    
tfidf_train_acc,tfidf_val_acc,tfidf_train_f1,tfidf_val_f1=support_metrics(tfidf_train_scores,tfidf_val_scores)

In [18]:
!pip install wandb -q

In [19]:
import wandb
from kaggle_secrets import UserSecretsClient
import os

secrets=UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
wandb.login()
wandb_mode=True

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [20]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="tfidf-baseline-metrics")
    wandb.log({
        "map_at_3_train":tfidf_train_map3,   
        "map_at_3_val":tfidf_val_map3,
        "accuracy_at_train":tfidf_train_acc,
        "accuracy_at_val":tfidf_val_acc,
        "f1_at_train":tfidf_train_f1,
        "f1_at_val":tfidf_val_f1,
        "vectorizer":"TfidfVectorizer",
        "sublinear_tf":True})
    wandb.finish()

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260622_205017-wvdxg1n4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run tfidf-baseline-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/wvdxg1n4
wandb: updating run metadata; uploading summary
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.33375
wandb:   accuracy_at_val 0.3375
wandb:       f1_at_train 0.33152
wandb:         f1_at_val 0.33492
wandb:    map_at_3_train 0.52438
wandb:      map_at_3_val 0.51333
wandb:      sublinear_tf True
wandb:        vectorizer TfidfVectorizer
wandb: 
wandb:

In [21]:
from gensim.models import Word2Vec
np.random.seed(1)
from numpy.random import seed
seed(1)

sentences=[text.split() for text in all_text]
word2vec=Word2Vec(sentences,vector_size=10,window=1,min_count=1,seed=1,workers=1,epochs=10)

In [22]:
def sentence_embedding(text):
    words=text.split()
    embedding=[word2vec.wv[word] for word in words if word in word2vec.wv]
    if len(embedding)==0:
        return np.zeros(word2vec.vector_size)
    return np.mean(embedding,axis=0)

In [23]:
prompts=np.vstack(x_train['prompt'].apply(sentence_embedding))
option_cols=['A','B','C','D','E']
w2v_train_scores=[]

for col in option_cols:
    option_matrix=np.vstack(x_train[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(prompts,option_matrix))
    w2v_train_scores.append(scores)
w2v_train_scores=np.array(w2v_train_scores).T

In [24]:
val_prompts=np.vstack(x_val['prompt'].apply(sentence_embedding))
w2v_val_scores=[]

for col in option_cols:
    option_matrix=np.vstack(x_val[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(val_prompts,option_matrix))
    w2v_val_scores.append(scores)
w2v_val_scores=np.array(w2v_val_scores).T

In [25]:
w2v_train_labels=score_to_labels(w2v_train_scores)
w2v_train_map3=map_at_3(w2v_train_labels,y_train)
print(w2v_train_map3)

w2v_val_labels=score_to_labels(w2v_val_scores)
w2v_val_map3=map_at_3(w2v_val_labels,y_val)
print(w2v_val_map3)

0.3823958333333334
0.38583333333333336


In [26]:
w2v_train_acc,w2v_val_acc,w2v_train_f1,w2v_val_f1=support_metrics(w2v_train_scores,w2v_val_scores)

if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="word2vec-baseline-metrics")
    wandb.log({
        "map_at_3_train":w2v_train_map3,   
        "map_at_3_val":w2v_val_map3,
        "accuracy_at_train":w2v_train_acc,
        "accuracy_at_val":w2v_val_acc,
        "f1_at_train":w2v_train_f1,
        "f1_at_val":w2v_val_f1,
        "embedding_model":"Word2Vec",
        "vector_size":20,
         "window":1,
         "min_count":1})
    wandb.finish()

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260622_205048-wv6x46q2
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run word2vec-baseline-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/wv6x46q2
wandb: updating run metadata; uploading summary
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb:         min_count ▁
wandb:       vector_size ▁
wandb:            window ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.185
wandb:   accuracy_at_val 0.175
wandb:   embedding_model Word2Vec
wandb:       f1_at_train 0.18457
wandb:         f1_at_val 0.17689
wandb:    map_at_3_train 0.3824
wand

In [27]:
from sklearn.preprocessing import StandardScaler

ss_tfidf=StandardScaler()
ss_w2v=StandardScaler()
train_final1=ss_tfidf.fit_transform(tfidf_train_scores)
train_final2=ss_w2v.fit_transform(w2v_train_scores)
val_final1=ss_tfidf.transform(tfidf_val_scores)
val_final2=ss_w2v.transform(w2v_val_scores)

In [28]:
def weighted_avg(final1,final2):
    best_score=0
    best_weights=[0.33,0.33,0.33]
    for first in np.arange(0,1.01,0.05):
        second=1-first
        if second<0:
            continue
        final=first*final1+second*final2
        labels=[]
        for score in final:
            scored=np.argsort(score)[::-1][:3]
            top_3=[chr(65+i) for i in scored]
            labels.append(" ".join(top_3)) 
        score_val = map_at_3(labels, y_train)
        if score_val>best_score:
            best_score=score_val
            best_weights=[first,second]
    return best_score,best_weights

best_score,best_weights=weighted_avg(train_final1,train_final2)
best_score

0.5522916666666666

In [29]:
final_train=best_weights[0]*train_final1+best_weights[1]*train_final2
final_val=best_weights[0]*val_final1+best_weights[1]*val_final2

final_train_labels=score_to_labels(final_train)
final_val_labels=score_to_labels(final_val)

comb_train_map3=map_at_3(final_train_labels,y_train)
comb_val_map3=map_at_3(final_val_labels,y_val)

print(map_at_3(final_train_labels,y_train))
print(map_at_3(final_val_labels,y_val))

0.5522916666666666
0.5229166666666667


In [30]:
comb_train_acc,comb_val_acc,comb_train_f1,comb_val_f1=support_metrics(final_train,final_val)

if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="word2vec_and_tfidf-baseline-metrics")
    wandb.log({
        "map_at_3_train":comb_train_map3,   
        "map_at_3_val":comb_val_map3,
        "accuracy_at_train":comb_train_acc,
        "accuracy_at_val":comb_val_acc,
        "f1_at_train":comb_train_f1,
        "f1_at_val":comb_val_f1,
        "embedding_model":"Word2Vec+TFIDF",
        "best_weights":best_weights})
    wandb.finish()

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260622_205050-4ciw460g
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run word2vec_and_tfidf-baseline-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/4ciw460g
wandb: updating run metadata; uploading summary
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.39125
wandb:   accuracy_at_val 0.38
wandb:   embedding_model Word2Vec+TFIDF
wandb:       f1_at_train 0.38682
wandb:         f1_at_val 0.37642
wandb:    map_at_3_train 0.55229
wandb:      map_at_3_val 0.52292
wandb: 
wandb: 🚀 View run word2vec

In [31]:
test=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

test['prompt']=test['prompt'].apply(clean_text)
test['A']=test['A'].apply(clean_text)
test['B']=test['B'].apply(clean_text)
test['C']=test['C'].apply(clean_text)
test['D']=test['D'].apply(clean_text)
test['E']=test['E'].apply(clean_text)

tfidf_test_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(test['prompt']+" "+test[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_test_scores.append(np.asarray(scores).flatten())
tfidf_test_scores=np.array(tfidf_test_scores).T

In [32]:
w2v_prompts=np.vstack(test['prompt'].apply(sentence_embedding))
test_w2v_scores=[]
for col in ['A','B','C','D','E']:
    option_matrix=np.vstack(test[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(w2v_prompts,option_matrix))
    test_w2v_scores.append(scores)

test_w2v_scores=np.array(test_w2v_scores).T

In [33]:
test_final1=ss_tfidf.transform(tfidf_test_scores)
test_final2=ss_w2v.transform(test_w2v_scores)

In [34]:
final_test=best_weights[0]*test_final1+best_weights[1]*test_final2
test_labels=[]
for score in final_test:
    scored=np.argsort(score)[::-1][:3]
    top_3=[chr(65+i) for i in scored]
    test_labels.append(" ".join(top_3))

In [35]:
submission=pd.DataFrame({'id':test['id'],'Prediction':test_labels})
submission.to_csv('submission_model1_w2v.csv',index=False)
print("Model submission saved!")
print(submission.head())

Model submission saved!
   id Prediction
0   1      C E B
1   2      E D C
2   3      E B C
3   4      D B C
4   5      D E B
